# SQL Alchemy Demo

In [2]:
from __future__ import annotations

import os
from sqlalchemy import create_engine, text, select
from sqlalchemy.orm import Session
from sqlalchemy.exc import SQLAlchemyError, IntegrityError
from model.Klient import Klient
from model.Schuld import Schuld
from model.Gläubiger import Gläubiger
from model.User import User

engine = create_engine(
    f'mysql+mysqlconnector://{os.getenv("DB_USER")}:{os.getenv("DB_PASSWORD")}@{os.getenv("DB_HOST")}/{os.getenv("DB_NAME")}',
    echo=True
)

### Datenbankverbindung via Engine
(Slide 8)

In [3]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT Vname, Nname FROM Klient"))
    
rows = result.fetchall()
print(rows)

2026-10-30 08:16:08,030 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-10-30 08:16:08,032 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-30 08:16:08,033 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-10-30 08:16:08,033 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-30 08:16:08,033 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-10-30 08:16:08,033 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-30 08:16:08,033 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-30 08:16:08,044 INFO sqlalchemy.engine.Engine SELECT Vname, Nname FROM Klient
2026-10-30 08:16:08,045 INFO sqlalchemy.engine.Engine [generated in 0.00219s] {}
2026-10-30 08:16:08,063 INFO sqlalchemy.engine.Engine ROLLBACK
[('Jyoti', 'Vyjajakulenthiran'), ('Heinz', 'Walter'), ('Mirjam', 'Kialla'), ('Nela', 'Keller'), ('Neda', 'Braun'), ('Baruch', 'Ahava'), ('Beat', 'Mayer'), ('Selina', 'Medebo'), ('Sybille', 'Jori'), ('Alexander', 'Baumann')]


### SQL Injection despite ORM
(Slide 8 + 9)

In [ ]:
# SQL Injection with textual SQL
user_id = '28 OR 1=1'
query = f"SELECT * FROM User WHERE UserId = {user_id}"

with Session(engine) as session:
    result = session.execute(text(query))
    for row in result:
        print(row)

In [ ]:
# SQLAlchemy impedes SQL injection with full ORM-based implementation
stmt = select(User).where(User.userId == user_id)
result = session.execute(stmt).scalar_one_or_none()

print(result.username, result.passwort)

### Unterschied zwischen execute() und scalars()
(Slide 8)

In [ ]:
with Session(engine) as session:
    result_1 = session.execute(select(Klient.vname, Klient.nname))

    for x in result_1:
        print(x)

In [ ]:
result_2 = session.scalars(select(Klient.nname, Klient.vname)).all()
for y in result_2:
    print(y)

### scalars() mit ganzem Objekt
(Slide 8)

In [ ]:
statement_1 = select(Klient).filter(Klient.klientenNr > 40000)
klienten_objekte = session.scalars(statement_1).all() # returns a list of Klient objects
for x in klienten_objekte:
    print(x)

In [ ]:
for x in klienten_objekte:
    print(f"Klient: {x.vname} {x.nname}")

### Objektinstanz in DB einfügen
(Slide 11 + 12)

In [ ]:
with Session(engine) as session:
    try:
        klient_obj = Klient(
            klientenNr=38201, 
            nname="Barrio", 
            vname="Eduardo", 
            strasse="Emmersmatt", 
            plz=3414, 
            ort="Rüdtligen-Alchenflüh", 
            telefonNr="0317511234", 
            beruf="arbeitslos", 
            monatseinkommen=0
            )

        session.add(klient_obj)
        session.flush()  # Flush the session to check for integrity errors before committing
    except IntegrityError as e:
        print(f"Object already exists in database: {e}")
        session.rollback()
    except SQLAlchemyError as e:
        print(f"An error occurred: {e}")
        session.rollback()
    else:
        session.commit()

### Lazy Loading & Repository Pattern
(Slide 14)

In [3]:
from repository.schuld_repo import SchuldRepository

with Session(engine) as session:
    sRepo = SchuldRepository(session)
    schuld = sRepo.getSchuldBySchuldNr(9547)
    print(f"SchuldNr: {schuld.schuldNr}, Höhe: {schuld.höhe}, GläubigerNr: {schuld.gläubigerNr}, KlientenNr: {schuld.klientenNr}, Grund: {schuld.grund}")

    print("--------------------------------------------------------------------------------------------------------------------------------------------")
    schuld.gläubiger  # Accessing the related Gläubiger object
    print(f"Gläubiger: {schuld.gläubiger.vname} {schuld.gläubiger.nname}")


2026-10-29 12:20:35,174 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-29 12:20:35,177 INFO sqlalchemy.engine.Engine SELECT `Schuld`.`SchuldNr` AS `Schuld_SchuldNr`, `Schuld`.`Höhe` AS `Schuld_Höhe`, `Schuld`.`GläubigerNr` AS `Schuld_GläubigerNr`, `Schuld`.`KlientenNr` AS `Schuld_KlientenNr`, `Schuld`.`Grund` AS `Schuld_Grund` 
FROM `Schuld` 
WHERE `Schuld`.`SchuldNr` = %(SchuldNr_1)s 
 LIMIT %(param_1)s
2026-10-29 12:20:35,178 INFO sqlalchemy.engine.Engine [generated in 0.00119s] {'SchuldNr_1': 9547, 'param_1': 1}
SchuldNr: 9547, Höhe: 44200, GläubigerNr: 26258, KlientenNr: 74105, Grund: Mieten nicht bezahlt
--------------------------------------------------------------------------------------------------------------------------------------------
2026-10-29 12:20:35,183 INFO sqlalchemy.engine.Engine SELECT `Gläubiger`.`GläubigerNr` AS `Gläubiger_GläubigerNr`, `Gläubiger`.`Nname` AS `Gläubiger_Nname`, `Gläubiger`.`Vname` AS `Gläubiger_Vname`, `Gläubiger`.`Strasse` AS `Gläubiger

### Manual Schema Generation with SQLAlchemy
(Slide 16 + 17)

In [4]:
from sqlalchemy import MetaData, Table, Column, Integer, String, inspect


inspector = inspect(engine)
metadata = MetaData()

if inspector.has_table("Seller"):
	print("Table 'Seller' exists")
else:
	print("Table 'Seller' does not exist")
	seller = Table(
    "Seller",
    metadata,
    Column("SellerNr", Integer, primary_key=True),
    Column("Nname", String(100)),
    Column("Vname", String(100))
	)

	metadata.create_all(engine)

2026-10-29 12:26:12,293 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-29 12:26:12,295 INFO sqlalchemy.engine.Engine DESCRIBE `schuldenberatung`.`Seller`
2026-10-29 12:26:12,296 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-29 12:26:12,305 INFO sqlalchemy.engine.Engine ROLLBACK
Table 'Seller' does not exist
2026-10-29 12:26:12,308 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-29 12:26:12,309 INFO sqlalchemy.engine.Engine DESCRIBE `schuldenberatung`.`Seller`
2026-10-29 12:26:12,310 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-29 12:26:12,311 INFO sqlalchemy.engine.Engine 
CREATE TABLE `Seller` (
	`SellerNr` INTEGER NOT NULL AUTO_INCREMENT, 
	`Nname` VARCHAR(100), 
	`Vname` VARCHAR(100), 
	PRIMARY KEY (`SellerNr`)
)


2026-10-29 12:26:12,312 INFO sqlalchemy.engine.Engine [no key 0.00108s] {}
2026-10-29 12:26:12,337 INFO sqlalchemy.engine.Engine COMMIT


### Reflection

In [ ]:
from sqlalchemy import inspect

inspector = inspect(engine)

tables = inspector.get_table_names()
print(tables)

if inspector.has_table("Klient"):
    columns = inspector.get_columns('Klient')
    for column in columns:
        print(f"Column: {column['name']}, Type: {column['type']}, Nullable: {column['nullable']}, Default: {column['default']}")
else:
    print("Table 'Klient' does not exist in the database.")

2026-10-30 08:26:19,088 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-30 08:26:19,090 INFO sqlalchemy.engine.Engine SHOW FULL TABLES FROM `schuldenberatung`
2026-10-30 08:26:19,091 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-30 08:26:19,093 INFO sqlalchemy.engine.Engine ROLLBACK
['gläubiger', 'klient', 'schuld', 'user', 'vertrag']
2026-10-30 08:26:19,095 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-30 08:26:19,095 INFO sqlalchemy.engine.Engine DESCRIBE `schuldenberatung`.`Klient`
2026-10-30 08:26:19,096 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-30 08:26:19,096 INFO sqlalchemy.engine.Engine ROLLBACK
2026-10-30 08:26:19,096 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-30 08:26:19,096 INFO sqlalchemy.engine.Engine SHOW CREATE TABLE `Klient`
2026-10-30 08:26:19,096 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-30 08:26:19,096 INFO sqlalchemy.engine.Engine ROLLBACK
Column: KlientenNr, Type: INTEGER, Nullable: False, Default: None
Column: Nna

In [ ]:
from sqlalchemy import MetaData, Table
from sqlalchemy import Table


metadata = MetaData()

klienten = Table("Klient", metadata, autoload_with=engine)

[k.name for k in klienten.columns]


In [ ]:
from sqlalchemy import MetaData
from sqlalchemy.orm import DeclarativeBase, Mapped, relationship

metadata = MetaData()
metadata.reflect(bind=engine, only=["klient", "schuld"])

class Base(DeclarativeBase):
    pass

class Klient(Base):
    __table__ = metadata.tables["klient"]

    schuld: Mapped[list["Schuld"]] = relationship(
        "Schuld",
        back_populates="klient",
        cascade="all, delete-orphan"
    )


class Schuld(Base):
    __table__ = metadata.tables["schuld"]

    klient: Mapped["Klient"] = relationship(
        "Klient",
        back_populates="schuld"
    )

In [ ]:
mapper = inspect(Schuld)

print(mapper.columns.keys())  # Print the column names of the Klient table
print(mapper.relationships.keys())  # Print the relationship names of the Klient table

In [ ]:
klient = Klient(
    KlientenNr=1,
    Nname="Muster",
    Vname="Max",
    Strasse="Bahnhofstrasse 1",
    PLZ=8001,
    Ort="Zürich",
    TelefonNr=123456789,
    Beruf="Entwickler",
    Monatseinkommen=6000
)

print(klient)
print(klient.KlientenNr)
print(klient.Nname)
print(klient.Vname)